In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import re
from scipy import stats
import mysql.connector

In [ ]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="avito_db"
)

cursor = conn.cursor()

In [ ]:
query = "SELECT * FROM bureaux_Transformed"
df = pd.read_sql(query, conn)

In [ ]:
df.head()

In [ ]:
df.columns

<h2 style="color:black"> Binary attributes Vizualisation </h2>

In [ ]:
attrs = ['Terrasse', 'Parking', 'Securite', 'Chauffage', 'Climatisation', 'Ascenseur']

fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(7, 10))  

for i, feat in enumerate(attrs):
    sns.countplot(x=feat, data=df, ax=axes[i // 2, i % 2],width = 0.4, color='orange')
    axes[i // 2, i % 2].set_xlabel(feat)
    axes[i // 2, i % 2].set_ylabel('Count')

plt.tight_layout()
plt.show()

<h2 style="color:black"> Numerical attributes Vizualisation </h2>

In [ ]:
features2 = ['Surface_Totale', 'Etage', 'Age_encoded',
       'Condition_encoded', 'Quartier_enc']

fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(10, 6))

for i, feat in enumerate(features2):
    ax = axes[i // 2, i % 2]
    
    if feat == "Surface_Totale":
        sns.histplot(df[feat], binwidth=50, color='orange', ax=ax)
        ax.set_xlim(0, 500)
    elif feat == "Quartier_enc":
        sns.histplot(df[feat], binwidth=10000, color='orange', ax=ax)
        ax.set_xlim(0, 2500000)
    else:
        sns.histplot(df[feat], binwidth=1, color='orange', ax=ax)
        ax.set_xlim(0, 8)
    
    ax.set_xlabel(feat)
    ax.set_ylabel('Frequency')

# Supprimer le 4e subplot vide si seulement 3 features
if len(features2) < 4:
    fig.delaxes(axes[1, 1])

plt.tight_layout()
plt.show()


In [ ]:
columns = ['Salles_De_Bain', 'Etage', 'Age_encoded',
       'Condition_encoded']
for col in columns:
    print(f"Value counts for {col}:")
    print(df[col].value_counts())

In [ ]:
features2 = ['Prix', 'Surface_Totale', 
        'Etage', 'Age_encoded','Condition_encoded', 'Quartier_enc']
new_df = df[features2].copy()

In [ ]:
correlation_matrix = new_df.corr()

plt.figure(figsize=(11,11))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix for numeric features in the dataset')
plt.xticks(fontsize=6)
plt.yticks(fontsize=6)
plt.show()

In [ ]:
features1 = ['Prix', 'Terrasse', 'Parking', 'Securite', 'Chauffage',
       'Climatisation', 'Ascenseur']
new_df_1 = df[features1].copy()

In [ ]:
# We use the Pearson Correlation method for the binary attributes
corr_matrix = new_df_1.corr(method='pearson')
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f',xticklabels=new_df_1.columns, yticklabels=new_df_1.columns)
plt.title('Correlation Matrix of Binary Features')
plt.show()

<h2 style="color:black"> Features selection:</h2>

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import pandas as pd
import matplotlib.pyplot as plt

# Imputer & scaler
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(df.drop(columns=['Prix', 'Quartier', 'Age', 'Condition', 'Lien']))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# PCA
pca = PCA(n_components=5)
X_pca = pca.fit_transform(X_scaled)

# DataFrame PCA
df_pca = pd.DataFrame(X_pca, columns=[f'PCA_Component_{i+1}' for i in range(5)])

# Ajouter Prix
df_final = pd.concat([df_pca, df['Prix'].reset_index(drop=True)], axis=1)

# Variance expliquée
print("Explained variance ratio:", pca.explained_variance_ratio_)

# Scatter plot : 1 vs 2
plt.scatter(df_pca['PCA_Component_1'], df_pca['PCA_Component_2'], color="#0080ff")
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('PCA Components Scatter Plot')
plt.show()


In [ ]:
import numpy as np

# Calcul de la variance cumulée
explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# Plot
plt.figure(figsize=(8,5))
plt.plot(range(1, len(explained_var)+1), cumulative_var, marker='o', linestyle='--', color='b')
plt.xlabel('Number of PCA Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Scree Plot: Cumulative Explained Variance')
plt.grid(True)
plt.show()


In [ ]:
import numpy as np

# Remplacer les vrais NaN (np.nan) ou chaînes "nan"/"NaN" par None
df = df.replace(["nan", "NaN", np.nan], None)

# Liste des colonnes sans 'id'
columns = [col for col in df.columns if col != "id"]

# Construction de la requête
cols_str = ", ".join(f"`{col}`" for col in columns)
placeholders = ", ".join(["%s"] * len(columns))
insert_query = f"INSERT INTO bureaux_Transformed_Explored ({cols_str}) VALUES ({placeholders})"

# Insertion dans MySQL
for _, row in df[columns].iterrows():
    cursor.execute(insert_query, tuple(row))

conn.commit()


In [ ]:
cursor.close()
conn.close()
